This Jupyter notebook is developed under CEC EPIC Contract No. EPC-22-001 (WARP to Resilience) by Lumen Energy Strategy. <br>

The script is used to compile hourly downscaled climate projections using Cal-Adapt's climakitae application. <br>
https://climakitae.readthedocs.io/en/latest/ <br>

1. <b> Air Temperature at 2m </b> localized at station level
2. <b> Air Temperature at 2m </b> for gridcell closest to each station
3. <b> Dew point temperature </b> for gridcell closest to each station
4. <b> Relative humidity </b> for gridcell closest to each station
5. <b> Precipitation (total) </b> for gridcell closest to each station
6. <b> Shortwave surface downward direct normal irradiance (DNI) </b> for gridcell closest to each station
7. <b> Shortwave surface downward diffuse irradiance (DHI) </b> for gridcell closest to each station
8. <b> Instantaneous downwelling shortwave flux at bottom (GHI) </b> for gridcell closest to each station
9. <b> Instantaneous downwelling clear sky shortwave flux at bottom (clearsky GHI)</b> for gridcell closest to each station
10. <b> Wind speed at 10m </b> for gridcell closest to each station

For questions, please contact: <br>
Onur Aydin (EPC-22-001 Principal Investigator) at onur@lumenenergystrategy.com and/or <br>
Mithra Moezzi (EPC-22-001 Commission Agreement Manager) at mithra.moezzi@energy.ca.gov

<i> *Please do not distribute without permission from Lumen Energy Strategy.

### Import libraries

In [ ]:
import pandas as pd
import xarray as xr
import climakitae as ck
from climakitae.core.data_interface import DataParameters
from climakitae.util.utils import get_closest_gridcell
import warnings
warnings.filterwarnings("ignore")

In [3]:
import climakitaegui as ckg
selections = ckg.Select()
selections.show()

### Define fuction to get hourly WRF data for selected inputs

In [4]:
def get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution):
    
    #select data options
    selections = DataParameters()
    selections.downscaling_method = 'Dynamical'
    selections.timescale = 'hourly'
    selections.variable = variable
    selections.units = units
    selections.data_type = data_type
    selections.scenario_historical=['Historical Climate']
    selections.scenario_ssp=['SSP 3-7.0']
    selections.time_slice = (1980, 2100)
    selections.resolution = f'{resolution} km'

    #load dataset
    if(data_type == 'Stations'):
        print(f'Downloading localized station-level {variable} data for {station_name}')
        selections.stations = [station_name]
        wrf_ds = selections.retrieve()
        wrf_ds = ck.load(wrf_ds)
    else:
        print(f'Downloading gridded {variable} data for {station_name}')
        selections.latitude = (lat-0.1, lat+0.1)
        selections.longitude = (lon-0.1, lon+0.1)
        selections.area_subset = 'none'
        selections.area_average = 'No'
        wrf_ds = selections.retrieve()
        wrf_ds = get_closest_gridcell(wrf_ds, lat, lon)
        wrf_ds = ck.load(wrf_ds)
        wrf_ds = wrf_ds.convert_calendar("noleap")
        wrf_ds = wrf_ds.chunk(dict(time=-1)).compute()
    
    #convert to dataframe
    wrf_df = wrf_ds.to_dataframe(dim_order=['time','simulation','scenario'])
    if(data_type == 'Gridded'): 
        wrf_df = wrf_df[[variable]].rename(columns={variable: station_name})
    wrf_df = wrf_df.unstack().unstack()
    wrf_df = wrf_df.sort_values(by=['time']).sort_index(axis=1)
    
    return wrf_df

### 1. Hourly localized temperatures at station level 

In [ ]:
variable = 'Air Temperature at 2m'
units = 'degF'
data_type = 'Stations'

station_list = pd.read_csv("inputs/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyTemp_SSP3-7.0_{station_name}_{resolution}km_extended_WRF_results.csv')

### 2. Hourly temperatures for gridcell closest to each station

In [6]:
variable = 'Air Temperature at 2m'
units = 'degF'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyTemp_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 3. Hourly dew point for gridcell closest to each station

In [ ]:
variable = 'Dew point temperature'
units = 'degF'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyDewPoint_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 4. Hourly relative humidity for gridcell closest to each station

In [ ]:
variable = 'Relative humidity'
units = '[0 to 100]'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyRelHumidity_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 5. Hourly precipitation for gridcell closest to each station

In [ ]:
variable = 'Precipitation (total)'
units = 'inches'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyPrecipitation_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 6. Hourly direct normal irradiance (DNI) for gridcell closest to each station

In [ ]:
variable = 'Shortwave surface downward direct normal irradiance'
units = 'W/m2'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyDNI_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 7. Hourly diffuse horizontal irradiance (DHI) for gridcell closest to each station

In [ ]:
### See if solar irradiance is actually used anywhere

In [ ]:
variable = 'Shortwave surface downward diffuse irradiance'
units = 'W/m2'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyDHI_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 8. Hourly global horizontal irradiance (GHI) for gridcell closest to each station

In [ ]:
variable = 'Instantaneous downwelling shortwave flux at bottom'
units = 'W/m2'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyGHI_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 9. Hourly clearsky global horizontal irradiance (clearsky GHI) for gridcell closest to each station

In [ ]:
variable = 'Instantaneous downwelling clear sky shortwave flux at bottom'
units = 'W/m2'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyClearskyGHI_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')

### 10. Hourly wind speed for gridcell closest to each station

In [ ]:
variable = 'Wind speed at 10m'
units = 'm s-1'
data_type = 'Gridded'

station_list = pd.read_csv("data/weather_stations.csv", index_col='station_name')

for station_name in station_list.index:
    lat = station_list.loc[station_name, 'LAT_Y']
    lon = station_list.loc[station_name, 'LON_X']
    resolution = station_list.loc[station_name, 'resolution']
    df_wrf = get_wrf_data (variable, units, data_type, station_name, lat, lon, resolution)
    df_wrf.to_csv(f'data/projections/hourlyWindSpeed_SSP3-7.0_{station_name}_closestgridcell_{resolution}km.csv')